In [1]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np

try:
    # Use raw string (r"...") or double backslashes to avoid unicode escape errors
    file_path = r"C:\Users\Lenovo\Downloads\global_ev_adoption_behavior_2026.csv"
    
    # Read CSV
    df = pd.read_csv(file_path)
    
    # Display shape
    print(f"Shape: {df.shape}")  # Example: (50000, 23)
    
    # Show data types
    print("\nData Types:")
    print(df.dtypes)
    
    # Check for missing values
    print("\nMissing Values per Column:")
    print(df.isnull().sum())

except FileNotFoundError:
    print("❌ Error: File not found. Please check the file path.")
except pd.errors.EmptyDataError:
    print("❌ Error: The file is empty.")
except pd.errors.ParserError as e:
    print(f"❌ Parsing Error: {e}")
except Exception as e:
    print(f"❌ Unexpected Error: {e}")


Shape: (50000, 23)

Data Types:
age                                 int64
annual_income                     float64
education_level                       str
city_type                             str
daily_commute_km                  float64
weekly_travel_distance_km         float64
current_vehicle_type                  str
vehicle_age_years                 float64
fuel_expense_per_month            float64
charging_station_accessibility    float64
nearest_charging_station_km       float64
home_charging_available             int64
electricity_cost_per_kwh          float64
environmental_awareness_score     float64
government_incentive_awareness    float64
technology_affinity_score         float64
range_anxiety_score               float64
battery_replacement_concern       float64
ev_knowledge_score                float64
previous_ev_experience              int64
ev_adoption_likelihood                str
monthly_energy_consumption_kwh    float64
monthly_charging_cost             float64
dt

In [6]:
# Count negatives before cleaning
neg_count = (df["fuel_expense_per_month"] < 0).sum()
print(f"Negative values: {neg_count}")
# Replace negatives with NaN
df["fuel_expense_per_month"] = df["fuel_expense_per_month"].where(
df["fuel_expense_per_month"] >= 0, other=np.nan
)
# Optional: fill NaN with median
median_fuel = df["fuel_expense_per_month"].median()
df["fuel_expense_clean"] = df["fuel_expense_per_month"].fillna(median_fuel)
print(f"Median fuel expense: ${median_fuel:.2f}")
print(f"Remaining nulls: {df['fuel_expense_per_month'].isnull().sum()}")

Negative values: 271
Median fuel expense: $295.60
Remaining nulls: 271


In [7]:
median_csa = df["charging_station_accessibility"].median()
print(f"Charging accessibility median: {median_csa}") # 5.9
df["charging_station_accessibility"] = (
df["charging_station_accessibility"].fillna(median_csa)
)
print(f"Nulls remaining: {df['charging_station_accessibility'].isnull().sum()}")

Charging accessibility median: 5.9
Nulls remaining: 0


In [8]:
median_evk = df["ev_knowledge_score"].median()
print(f"EV knowledge median: {median_evk}") # 7.0
df["ev_knowledge_score"] = df["ev_knowledge_score"].fillna(median_evk)
print(f"Nulls remaining: {df['ev_knowledge_score'].isnull().sum()}")

EV knowledge median: 7.0
Nulls remaining: 0


In [9]:
df["Home_Charging"] = df["home_charging_available"].map({1: "Yes", 0: "No"})
df["Prev_EV_Exp"] = df["previous_ev_experience"].map({1: "Yes", 0: "No"})
print(df["Home_Charging"].value_counts())
# Yes 32384
# No 17616

Home_Charging
Yes    32488
No     17512
Name: count, dtype: int64


In [10]:
bins = [20, 29, 39, 49, 59, 70]
labels = ["21-29", "30-39", "40-49", "50-59", "60+"]
df["Age_Group"] = pd.cut(df["age"], bins=bins, labels=labels)
print(df["Age_Group"].value_counts().sort_index())

Age_Group
21-29     9120
30-39    10143
40-49    10348
50-59    10296
60+      10093
Name: count, dtype: int64


In [17]:
def income_segment(income):
    if income < 25000:
        return "Low (<$25K)"
    elif income < 50000:
        return "Middle ($25K-$50K)"
    elif income < 100000:
        return "Upper Middle ($50K-$100K)"
    else:
        return "High (>$100K)"

# Apply the function
df["Income_Segment"] = df["annual_income"].apply(income_segment)
print(df["Income_Segment"].value_counts())

Income_Segment
Low (<$25K)                  2
Middle ($25K-$50K)           2
Upper Middle ($50K-$100K)    2
High (>$100K)                1
Name: count, dtype: int64


In [18]:
print(df.columns.tolist())

['annual_income', 'Income_Segment']


In [21]:
df["Potential_Monthly_Savings"] = (
df["fuel_expense_per_month"] - df["monthly_charging_cost"]
)
avg_savings = df["Potential_Monthly_Savings"].mean()
print(f"Average monthly savings if switched to EV: ${avg_savings:.2f}")

Average monthly savings if switched to EV: $254.93


In [22]:
order_map = {"Low": 1, "Medium": 2, "High": 3}
df["Adoption_Sort_Order"] = df["ev_adoption_likelihood"].map(order_map)

In [23]:
# Validate distributions
print("=== EV Adoption Likelihood ===")
print(df["ev_adoption_likelihood"].value_counts(normalize=True).round(3) * 100)
print("\n=== City Type ===")
print(df["city_type"].value_counts(normalize=True).round(3) * 100)
print("\n=== Nulls after cleaning ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
# Export clean CSV
df.to_csv("ev_adoption_cleaned.csv", index=False)
print("\nExported: ev_adoption_cleaned.csv")
print(f"Final shape: {df.shape}") # (50000, 29) — 6 new columns added

=== EV Adoption Likelihood ===
ev_adoption_likelihood
High      59.3
Medium    24.2
Low       16.5
Name: proportion, dtype: float64

=== City Type ===
city_type
Urban       45.3
Suburban    34.7
Rural       20.0
Name: proportion, dtype: float64

=== Nulls after cleaning ===
education_level                   500
charging_station_accessibility    500
ev_knowledge_score                500
dtype: int64

Exported: ev_adoption_cleaned.csv
Final shape: (50000, 25)
